In [2]:
import random
import time
from typing import List, Dict, Optional, Tuple, Set
from dataclasses import dataclass
from enum import Enum


def generate_mac_address() -> str:
    """
    Generate a random MAC address.
    
    Format: XX:XX:XX:XX:XX:XX (6 bytes, each in hex 00-FF)
    Example: "a4:5e:60:8f:72:1b"
    
    Why 6 bytes? IEEE 802 standard specifies 48-bit MAC addresses.
    First 3 bytes: Manufacturer ID (OUI)
    Last 3 bytes: Unique device ID
    """
    # Generate 6 random bytes, format as hex with colons
    bytes_list = [f"{random.randint(0, 255):02x}" for _ in range(6)]
    return ":".join(bytes_list)


# Test MAC generation
print("=" * 60)
print("STEP 1: MAC ADDRESS GENERATION")
print("=" * 60)
for i in range(3):
    mac = generate_mac_address()
    print(f"Generated MAC {i+1}: {mac}")
print(f"\nEvery network device has a unique MAC address burned into its hardware.")
print(f"This is like a unique serial number that never changes.")

STEP 1: MAC ADDRESS GENERATION
Generated MAC 1: f9:c5:c2:ba:c2:5c
Generated MAC 2: 20:1e:32:f2:7d:a8
Generated MAC 3: ad:20:cb:e3:b3:9d

Every network device has a unique MAC address burned into its hardware.
This is like a unique serial number that never changes.


In [3]:
class EndDevice:
    """
    Enhanced End Device with MAC Address (Layer 2 ready).
    
    In Layer 1, we only had names. Now each device has a unique
    hardware identifier (MAC address) that switches can learn.
    """
    
    def __init__(self, name: str):
        self.name = name
        # Layer 1 attributes
        self.connections: List['Device'] = []  # Physical connections
        self.inbox: List['Frame'] = []         # Received frames (not raw packets anymore)
        
        # Layer 2 attributes
        self.mac_address: str = generate_mac_address()  # Unique hardware ID
        self.is_promiscuous: bool = False  # If True, accepts all frames (like Wireshark)
        
        print(f"[CREATED] {self.name} with MAC {self.mac_address}")
    
    def connect(self, other: 'Device'):
        """Create bidirectional physical connection."""
        if other not in self.connections:
            self.connections.append(other)
            if self not in other.connections:
                other.connections.append(self)
            print(f"[CONNECT] {self.name} ↔ {other.name}")
    
    def send(self, frame: 'Frame'):
        """
        Send frame to all connected devices.
        In real networks, this would be electrical signals.
        """
        print(f"\n[SEND] {self.name} ({self.mac_address}) sending:")
        print(f"       {frame}")
        
        for device in self.connections:
            device.receive(frame, self)
    
    def receive(self, frame: 'Frame', from_device: 'Device'):
        """
        Receive a frame. Check if it's for us by MAC address.
        
        Layer 2 improvement: Check destination MAC, not just name!
        """
        print(f"  [RECV] {self.name} got frame from {from_device.name}")
        
        # Check if frame is for us (or broadcast)
        is_for_me = (
            frame.dest_mac == self.mac_address or      # Specifically for me
            frame.dest_mac == "FF:FF:FF:FF:FF:FF" or  # Broadcast address
            self.is_promiscuous                        # Sniffing mode (Wireshark)
        )
        
        if is_for_me:
            print(f"    ✓ ACCEPTED: MAC {frame.dest_mac} matches me (or broadcast)")
            self.inbox.append(frame)
            return True
        else:
            print(f"    ✗ IGNORED: MAC {frame.dest_mac} is not my MAC ({self.mac_address})")
            return False
    
    def __str__(self):
        return f"{self.name}[{self.mac_address}]"


# Frame class (Layer 2 data unit)
@dataclass
class Frame:
    """
    Layer 2 Frame (replaces Layer 1 Packet).
    
    Layer 1: Packet (just data)
    Layer 2: Frame (data + MAC addresses + error checking)
    
    Structure:
    | Destination MAC | Source MAC | Data | Error Check |
    |    6 bytes      |  6 bytes   |  ...  |   4 bytes   |
    """
    dest_mac: str      # Who should receive this (MAC address)
    src_mac: str       # Who sent this (MAC address)
    data: str          # Actual payload
    seq_num: int = 0   # For flow control (sequence number)
    crc: str = ""      # Error detection code
    
    def __str__(self):
        return f"Frame({self.src_mac} → {self.dest_mac}): '{self.data}' [CRC:{self.crc}]"



## testing end devices 

In [4]:

# Test enhanced EndDevice
print("\n" + "=" * 60)
print("STEP 2: ENHANCED END DEVICE WITH MAC ADDRESS")
print("=" * 60)

pc1 = EndDevice("PC1")
pc2 = EndDevice("PC2")

print(f"\nPC1 MAC: {pc1.mac_address}")
print(f"PC2 MAC: {pc2.mac_address}")

# Connect and send
pc1.connect(pc2)

# Create frame with MAC addresses
frame = Frame(
    dest_mac=pc2.mac_address,  # Specifically addressed to PC2
    src_mac=pc1.mac_address,   # From PC1
    data="Hello PC2!"
)

pc1.send(frame)

print(f"\n[RESULT] PC2 inbox has {len(pc2.inbox)} frame(s)")
if pc2.inbox:
    print(f"         Content: {pc2.inbox[0].data}")


STEP 2: ENHANCED END DEVICE WITH MAC ADDRESS
[CREATED] PC1 with MAC 42:17:b8:ef:df:01
[CREATED] PC2 with MAC 43:13:42:6d:9b:20

PC1 MAC: 42:17:b8:ef:df:01
PC2 MAC: 43:13:42:6d:9b:20
[CONNECT] PC1 ↔ PC2

[SEND] PC1 (42:17:b8:ef:df:01) sending:
       Frame(42:17:b8:ef:df:01 → 43:13:42:6d:9b:20): 'Hello PC2!' [CRC:]
  [RECV] PC2 got frame from PC1
    ✓ ACCEPTED: MAC 43:13:42:6d:9b:20 matches me (or broadcast)

[RESULT] PC2 inbox has 1 frame(s)
         Content: Hello PC2!


## Implementing Switch

In [5]:
class Switch:
    """
    SWITCH - Layer 2 (Data Link Layer) Device.
    
    Key difference from Hub:
    - Hub: Dumb repeater, broadcasts everything to everyone
    - Switch: Smart device, learns MAC addresses, sends only to correct port
    
    How it works:
    1. Learning: When a frame arrives, note "Source MAC came from Port X"
    2. Forwarding: Look up Destination MAC, send only to that port
    3. Unknown: If MAC not known, broadcast (like hub) - but only once!
    
    Real-world analogy: Smart mailroom clerk with a directory.
    """
    
    def __init__(self, name: str, num_ports: int = 8):
        self.name = name
        self.num_ports = num_ports
        
        # MAC Address Table (CAM Table)
        # Maps: MAC Address → (Device, Port Number)
        # This is the "brain" of the switch!
        self.mac_table: Dict[str, Tuple[Device, int]] = {}
        
        # Track which device is on which port
        self.port_map: Dict[Device, int] = {}
        self.connections: List[Device] = []
        
        print(f"[CREATED] Switch {self.name} with {num_ports} ports")
    
    def connect(self, device: 'Device', port: int = None):
        """
        Connect a device to a specific port on the switch.
        
        In real switches, ports are numbered (Port 1, Port 2, etc.).
        We track this for the MAC table.
        """
        if len(self.connections) >= self.num_ports:
            print(f"[ERROR] {self.name}: No free ports!")
            return
        
        # Assign port number if not specified
        if port is None:
            port = len(self.connections) + 1
        
        self.connections.append(device)
        self.port_map[device] = port
        
        # Also add switch to device's connections
        if self not in device.connections:
            device.connections.append(self)
        
        print(f"[CONNECT] {device.name} → {self.name} Port {port}")
    
    def receive(self, frame: Frame, from_device: 'Device'):
        """
        Main switch logic: Learn source, lookup destination, forward.
        
        This is called when any device sends a frame to the switch.
        """
        # Get the port number this frame came from
        in_port = self.port_map.get(from_device, 0)
        
        print(f"\n[SWITCH] {self.name} received frame on Port {in_port}")
        print(f"         Frame: {frame}")
        
        # STEP 1: LEARN SOURCE MAC
        # "I learned that frame.src_mac is on Port in_port"
        self._learn_mac(frame.src_mac, from_device, in_port)
        
        # STEP 2: FORWARD BASED ON DESTINATION MAC
        if frame.dest_mac == "FF:FF:FF:FF:FF:FF":
            # Broadcast frame - send to everyone except sender
            print(f"         [BROADCAST FRAME] Flooding to all ports")
            self._broadcast(frame, from_device)
        else:
            # Unicast frame - lookup in MAC table
            self._forward_unicast(frame, from_device)
    
    def _learn_mac(self, mac: str, device: 'Device', port: int):
        """
        LEARNING PHASE: Add/update MAC address table.
        
        If we see a frame from MAC address X on Port Y,
        we learn that X is reachable via Port Y.
        """
        if mac not in self.mac_table:
            print(f"         [LEARNED] New MAC {mac} on Port {port}")
        else:
            # Update if moved (MAC moved to different port)
            old_port = self.mac_table[mac][1]
            if old_port != port:
                print(f"         [UPDATED] MAC {mac} moved from Port {old_port} to Port {port}")
        
        self.mac_table[mac] = (device, port)
    
    def _forward_unicast(self, frame: Frame, from_device: 'Device'):
        """
        FORWARDING PHASE: Send only to the correct port.
        
        If we know where destination MAC is, send only there.
        If unknown, broadcast (flooding) - but only for unknown!
        """
        dest_mac = frame.dest_mac
        
        if dest_mac in self.mac_table:
            # We know where this MAC is! Forward directly.
            target_device, target_port = self.mac_table[dest_mac]
            print(f"         [FORWARD] Known MAC! Sending to Port {target_port} ({target_device.name})")
            target_device.receive(frame, self)
        else:
            # Unknown MAC - must broadcast (flooding)
            print(f"         [UNKNOWN MAC] {dest_mac} not in table, broadcasting to all...")
            self._broadcast(frame, from_device)
    
    def _broadcast(self, frame: Frame, exclude_device: 'Device'):
        """Send frame to all connected devices except the sender."""
        for device in self.connections:
            if device != exclude_device:
                print(f"         [FLOOD] → {device.name}")
                device.receive(frame, self)
    
    def print_mac_table(self):
        """Display the MAC address table - shows switch intelligence."""
        print(f"\n[MAC TABLE for {self.name}]")
        print("-" * 60)
        print(f"{'MAC Address':<20} {'Port':<6} {'Device':<15}")
        print("-" * 60)
        for mac, (device, port) in self.mac_table.items():
            print(f"{mac:<20} {port:<6} {device.name:<15}")
        print("-" * 60)
        print(f"Total entries: {len(self.mac_table)}")


# Base class for type hints
class Device:
    """Base class for any network device."""
    def __init__(self, name: str):
        self.name = name
        self.connections: List['Device'] = []
    
    def receive(self, frame: Frame, from_device: 'Device'):
        pass



## testing for switch

In [6]:

# Test Switch
print("\n" + "=" * 60)
print("STEP 3: SWITCH WITH MAC LEARNING")
print("=" * 60)

# Create switch and 3 PCs
switch = Switch("Switch1", num_ports=4)
pc_a = EndDevice("PC-A")
pc_b = EndDevice("PC-B")
pc_c = EndDevice("PC-C")

# Connect to switch ports
switch.connect(pc_a, port=1)
switch.connect(pc_b, port=2)
switch.connect(pc_c, port=3)

print("\n--- Test 1: PC-A sends to PC-B (Switch learns) ---")
frame1 = Frame(
    dest_mac=pc_b.mac_address,
    src_mac=pc_a.mac_address,
    data="Hello PC-B!"
)
pc_a.send(frame1)

switch.print_mac_table()

print("\n--- Test 2: PC-C sends to PC-A (Switch knows both now) ---")
frame2 = Frame(
    dest_mac=pc_a.mac_address,
    src_mac=pc_c.mac_address,
    data="Hi PC-A!"
)
pc_c.send(frame2)

switch.print_mac_table()

print("\n--- Test 3: PC-A sends to unknown MAC (Broadcast) ---")
fake_mac = "aa:bb:cc:dd:ee:ff"
frame3 = Frame(
    dest_mac=fake_mac,  # Switch doesn't know this MAC
    src_mac=pc_a.mac_address,
    data="Where are you?"
)
pc_a.send(frame3)


STEP 3: SWITCH WITH MAC LEARNING
[CREATED] Switch Switch1 with 4 ports
[CREATED] PC-A with MAC 20:18:9c:7b:a7:43
[CREATED] PC-B with MAC a4:e1:d6:cb:b8:8b
[CREATED] PC-C with MAC f5:75:7d:17:3e:e2
[CONNECT] PC-A → Switch1 Port 1
[CONNECT] PC-B → Switch1 Port 2
[CONNECT] PC-C → Switch1 Port 3

--- Test 1: PC-A sends to PC-B (Switch learns) ---

[SEND] PC-A (20:18:9c:7b:a7:43) sending:
       Frame(20:18:9c:7b:a7:43 → a4:e1:d6:cb:b8:8b): 'Hello PC-B!' [CRC:]

[SWITCH] Switch1 received frame on Port 1
         Frame: Frame(20:18:9c:7b:a7:43 → a4:e1:d6:cb:b8:8b): 'Hello PC-B!' [CRC:]
         [LEARNED] New MAC 20:18:9c:7b:a7:43 on Port 1
         [UNKNOWN MAC] a4:e1:d6:cb:b8:8b not in table, broadcasting to all...
         [FLOOD] → PC-B
  [RECV] PC-B got frame from Switch1
    ✓ ACCEPTED: MAC a4:e1:d6:cb:b8:8b matches me (or broadcast)
         [FLOOD] → PC-C
  [RECV] PC-C got frame from Switch1
    ✗ IGNORED: MAC a4:e1:d6:cb:b8:8b is not my MAC (f5:75:7d:17:3e:e2)

[MAC TABLE for Switch

## Implementing CRC


In [7]:
class ErrorControl:
    """
    ERROR CONTROL - Detecting corrupted data.
    
    Real problem: Electrical interference can flip bits (0→1 or 1→0).
    Solution: Add redundant check bits to detect errors.
    
    We'll implement CRC-4 (simple version) for demonstration.
    Real Ethernet uses CRC-32.
    """
    
    @staticmethod
    def calculate_crc(data_bits: str, polynomial: str = "1011") -> str:
        """
        Calculate CRC using polynomial division (XOR operations).
        
        How it works:
        1. Append (n-1) zeros to data (n = length of polynomial)
        2. Divide by polynomial using XOR
        3. Remainder is the CRC
        
        Example:
        Data: 1101, Polynomial: 1011 (x³ + x + 1)
        Step 1: 1101000 (append 3 zeros)
        Step 2: Divide 1101000 by 1011 using XOR
        Step 3: Remainder = 010 (CRC)
        """
        n = len(polynomial)
        # Append n-1 zeros
        dividend = list(data_bits + "0" * (n - 1))
        
        # Polynomial division
        for i in range(len(data_bits)):
            if dividend[i] == "1":
                # XOR with polynomial at this position
                for j in range(n):
                    # XOR: 1⊕1=0, 1⊕0=1, 0⊕1=1, 0⊕0=0
                    dividend[i + j] = "1" if dividend[i + j] != polynomial[j] else "0"
        
        # Remainder is last n-1 bits
        remainder = "".join(dividend[-(n-1):])
        return remainder
    
    @staticmethod
    def add_crc(frame: Frame, polynomial: str = "1011") -> Frame:
        """
        Add CRC to frame before sending.
        
        Convert data string to binary, calculate CRC, append it.
        """
        # Convert text data to binary string (simplified)
        data_bits = "".join(format(ord(c), "08b") for c in frame.data)
        
        # Calculate CRC
        crc = ErrorControl.calculate_crc(data_bits, polynomial)
        frame.crc = crc
        
        print(f"[CRC] Added CRC {crc} to frame")
        return frame
    
    @staticmethod
    def verify_crc(frame: Frame, polynomial: str = "1011") -> bool:
        """
        Verify frame integrity on receipt.
        
        Recalculate CRC on received data. If it matches the CRC in frame,
        data is intact. If not, corruption detected!
        """
        if not frame.crc:
            return True  # No CRC to check
        
        # Recalculate
        data_bits = "".join(format(ord(c), "08b") for c in frame.data)
        calculated_crc = ErrorControl.calculate_crc(data_bits, polynomial)
        
        is_valid = (calculated_crc == frame.crc)
        
        if is_valid:
            print(f"[CRC] ✓ Verification passed (CRC {frame.crc} matches)")
        else:
            print(f"[CRC] ✗ CORRUPTION DETECTED! Expected {calculated_crc}, got {frame.crc}")
        
        return is_valid
    
    @staticmethod
    def introduce_noise(data: str, error_rate: float = 0.1) -> str:
        """
        Simulate electrical noise flipping random bits.
        
        error_rate: Probability of each bit flipping (0.1 = 10%)
        """
        result = []
        flipped = 0
        
        for bit in data:
            if random.random() < error_rate and bit in "01":
                flipped += 1
                result.append("0" if bit == "1" else "1")
            else:
                result.append(bit)
        
        if flipped > 0:
            print(f"[NOISE] Introduced {flipped} bit error(s)")
        
        return "".join(result)



## Testing Error Control

In [8]:

# Test Error Control
print("\n" + "=" * 60)
print("STEP 4: ERROR CONTROL (CRC)")
print("=" * 60)

# Create frame
test_frame = Frame(
    dest_mac="aa:bb:cc:dd:ee:ff",
    src_mac="11:22:33:44:55:66",
    data="Hello"
)

print(f"Original frame: {test_frame}")

# Add CRC
ErrorControl.add_crc(test_frame)
print(f"With CRC: {test_frame}")

# Verify clean
print("\n--- Verification (clean) ---")
ErrorControl.verify_crc(test_frame)

# Simulate corruption
print("\n--- Simulating corruption ---")
# Corrupt the data (but keep CRC as-is to show detection)
original_char = test_frame.data[0]
test_frame.data = "X" + test_frame.data[1:]  # Change first character
print(f"Corrupted data: '{test_frame.data}' (was '{original_char}...')")

# Try to verify
ErrorControl.verify_crc(test_frame)


STEP 4: ERROR CONTROL (CRC)
Original frame: Frame(11:22:33:44:55:66 → aa:bb:cc:dd:ee:ff): 'Hello' [CRC:]
[CRC] Added CRC 100 to frame
With CRC: Frame(11:22:33:44:55:66 → aa:bb:cc:dd:ee:ff): 'Hello' [CRC:100]

--- Verification (clean) ---
[CRC] ✓ Verification passed (CRC 100 matches)

--- Simulating corruption ---
Corrupted data: 'Xello' (was 'H...')
[CRC] ✗ CORRUPTION DETECTED! Expected 010, got 100


False

## Flow Control

In [9]:
import random
import time


class CSMA_CD:
    """
    CSMA/CD - Carrier Sense Multiple Access with Collision Detection.
    
    CRITICAL: The channel is a SHARED medium. All devices on the same
    network segment must see the same channel state.
    """
    
    # CLASS VARIABLES (shared by ALL instances)
    # These represent the PHYSICAL MEDIUM that all devices share
    _channel_busy = False           # Is someone currently transmitting?
    _collision_detected = False     # Was a collision detected?
    _transmitting_devices = set()   # Which devices are currently sending?
    
    # Lock to prevent race conditions in simulation
    _lock = False
    
    def __init__(self, device_name: str):
        """
        Each device gets its own CSMA/CD instance,
        but they all share the same physical channel.
        """
        self.device_name = device_name
        self.collision_count = 0
        self.max_attempts = 16
    
    def transmit(self, data: str) -> bool:
        """
        Attempt to transmit data using CSMA/CD protocol.
        
        Returns: True if successful, False if max retries exceeded
        """
        attempt = 0
        
        while attempt < self.max_attempts:
            # Step 1: CARRIER SENSE (Listen before talking)
            # Check the SHARED channel state (class variable)
            if CSMA_CD._channel_busy:
                print(f"[CSMA/CD] {self.device_name}: Channel busy (someone else transmitting), listening...")
                time.sleep(0.05)
                continue
            
            # Check for collision detection flag
            if CSMA_CD._collision_detected:
                print(f"[CSMA/CD] {self.device_name}: Collision detected recently, waiting...")
                time.sleep(0.05)
                continue
            
            # Channel idle! Try to acquire it
            if not self._acquire_channel():
                # Someone else grabbed it just now
                continue
            
            # Step 2: Start transmitting
            print(f"[CSMA/CD] {self.device_name}: Acquired channel, transmitting {len(data)} bits...")
            
            # Simulate transmission time
            transmission_time = len(data) * 0.01
            time.sleep(transmission_time)
            
            # Step 3: COLLISION DETECTION
            # During transmission, check if someone else also started
            collision = self._detect_collision()
            
            if collision:
                # COLLISION! Stop immediately
                print(f"[CSMA/CD] {self.device_name}: COLLISION DETECTED during transmission!")
                self._handle_collision()
                attempt += 1
                
                # Binary Exponential Backoff
                backoff_time = self._calculate_backoff(attempt)
                print(f"[CSMA/CD] {self.device_name}: Backing off for {backoff_time*1000:.1f}ms (attempt {attempt})")
                time.sleep(backoff_time)
                
                # Retry
                continue
            
            # Success! Transmission complete
            self._release_channel()
            print(f"[CSMA/CD] {self.device_name}: Transmission successful!")
            self.collision_count = 0
            return True
        
        # Max attempts reached
        print(f"[CSMA/CD] {self.device_name}: FAILED after {self.max_attempts} attempts")
        return False
    
    def _acquire_channel(self) -> bool:
        """
        Try to acquire the shared channel.
        Returns True if acquired, False if someone else got it first.
        """
        # In real hardware: check voltage levels
        # In simulation: check if already busy
        if CSMA_CD._channel_busy:
            return False
        
        CSMA_CD._channel_busy = True
        CSMA_CD._transmitting_devices.add(self.device_name)
        return True
    
    def _detect_collision(self) -> bool:
        """
        Detect if collision occurred during transmission.
        
        Collision happens if:
        1. Another device started transmitting while we were
        2. (In real life: voltage levels indicate multiple signals)
        """
        # Simulate: 30% chance of collision if multiple devices want to send
        # In reality, collision occurs when two devices transmit simultaneously
        
        # Check if somehow channel state changed (another device interfered)
        # For simulation, we use random chance to represent contention
        if len(CSMA_CD._transmitting_devices) > 1:
            # Multiple devices transmitting = collision!
            return True
        
        # Simulate random noise/collision
        if random.random() < 0.2:  # 20% collision chance for demo
            # Simulate another "virtual" device causing collision
            return True
        
        return False
    
    def _handle_collision(self):
        """Handle collision: send jam signal, release channel."""
        # Send jam signal (32-bit pattern to ensure everyone detects collision)
        print(f"[CSMA/CD] {self.device_name}: Sending jam signal (32 bits)...")
        
        # Release channel immediately
        self._release_channel()
        
        # Set collision detected flag (others should wait)
        CSMA_CD._collision_detected = True
        self.collision_count += 1
        
        # Clear collision flag after short time (jam propagation)
        time.sleep(0.01)
        CSMA_CD._collision_detected = False
    
    def _release_channel(self):
        """Release the shared channel."""
        CSMA_CD._channel_busy = False
        CSMA_CD._transmitting_devices.discard(self.device_name)
    
    def _calculate_backoff(self, attempt: int) -> float:
        """
        Binary Exponential Backoff.
        
        After collision, wait random time before retry.
        Wait time increases exponentially with each attempt.
        
        Formula: Random(0, 2^min(attempt, 10) - 1) * slot_time
        """
        max_slots = (2 ** min(attempt, 10)) - 1
        slots = random.randint(0, max_slots)
        slot_time = 0.001  # 1 millisecond per slot (simplified)
        
        return slots * slot_time
    
    @classmethod
    def get_channel_status(cls) -> str:
        """Check current channel state (for debugging)."""
        status = "BUSY" if cls._channel_busy else "IDLE"
        transmitters = ", ".join(cls._transmitting_devices) if cls._transmitting_devices else "None"
        return f"Channel: {status}, Transmitting: {transmitters}"




In [13]:

# ============================================================
# CORRECTED TEST: Multiple Devices Sharing One Channel
# ============================================================

def test_csma_cd_shared_channel():
    """
    Test CSMA/CD with MULTIPLE devices sharing the SAME channel.
    
    Key point: All devices must use the shared class variables,
    not instance variables!
    """
    print("\n" + "=" * 70)
    print("CORRECTED CSMA/CD TEST: Shared Channel")
    print("=" * 70)
    print("""
    Scenario: 3 PCs connected to a shared medium (like a hub or bus).
    They all share the same physical channel - only one can transmit at a time.
    If two transmit simultaneously, collision occurs!
    """)
    
    # Create 3 devices - each has own CSMA_CD instance
    # BUT they all share the same channel (class variables)!
    pc_a = CSMA_CD("PC-A")
    pc_b = CSMA_CD("PC-B")
    pc_c = CSMA_CD("PC-C")
    
    print(f"\nInitial channel status: {CSMA_CD.get_channel_status()}")
    
    # Test 1: Sequential transmission (should work)
    print("\n" + "-" * 50)
    print("TEST 1: Sequential transmission")
    print("-" * 50)
    
    for pc in [pc_a, pc_b, pc_c]:
        print(f"\n>>> {pc.device_name} wants to send:")
        success = pc.transmit("10101010")
        print(f"Result: {'SUCCESS' if success else 'FAILED'}")
        print(f"Channel status: {CSMA_CD.get_channel_status()}")
    
    # Test 2: Simulated contention (may cause collisions)
    print("\n" + "-" * 50)
    print("TEST 2: Contention (random collisions)")
    print("-" * 50)
    
    # Reset collision counts
    for pc in [pc_a, pc_b, pc_c]:
        pc.collision_count = 0
    
    # Each tries to send again (may collide)
    results = {}
    for pc in [pc_a, pc_b, pc_c]:
        print(f"\n>>> {pc.device_name} wants to send again:")
        success = pc.transmit("11110000")
        results[pc.device_name] = success
    
    print("\n" + "-" * 50)
    print("SUMMARY")
    print("-" * 50)
    for name, success in results.items():
        status = "✓ Success" if success else "✗ Failed"
        print(f"  {name}: {status}")
    
    print(f"\nFinal channel status: {CSMA_CD.get_channel_status()}")
    
    # Verify channel is properly released
    assert not CSMA_CD._channel_busy, "ERROR: Channel still busy after all transmissions!"
    assert len(CSMA_CD._transmitting_devices) == 0, "ERROR: Devices still marked as transmitting!"
    print("\n✓ Channel properly released after all transmissions")



In [14]:


test_csma_cd_shared_channel()


CORRECTED CSMA/CD TEST: Shared Channel

    Scenario: 3 PCs connected to a shared medium (like a hub or bus).
    They all share the same physical channel - only one can transmit at a time.
    If two transmit simultaneously, collision occurs!
    

Initial channel status: Channel: IDLE, Transmitting: None

--------------------------------------------------
TEST 1: Sequential transmission
--------------------------------------------------

>>> PC-A wants to send:
[CSMA/CD] PC-A: Acquired channel, transmitting 8 bits...
[CSMA/CD] PC-A: Transmission successful!
Result: SUCCESS
Channel status: Channel: IDLE, Transmitting: None

>>> PC-B wants to send:
[CSMA/CD] PC-B: Acquired channel, transmitting 8 bits...
[CSMA/CD] PC-B: Transmission successful!
Result: SUCCESS
Channel status: Channel: IDLE, Transmitting: None

>>> PC-C wants to send:
[CSMA/CD] PC-C: Acquired channel, transmitting 8 bits...
[CSMA/CD] PC-C: COLLISION DETECTED during transmission!
[CSMA/CD] PC-C: Sending jam signal (32 b

In [15]:
class SlidingWindowProtocol:
    """
    SLIDING WINDOW PROTOCOL - Go-Back-N variant.
    
    Problem: Sender might send faster than receiver can process.
    Solution: Receiver grants "window" - permission to send N frames.
    
    Analogy: 
    - Window size = 3 means "You can send 3 emails without waiting"
    - After 3, wait for acknowledgments (replies)
    - When ACK received, slide window forward and send more
    
    Go-Back-N: If frame lost, resend from that frame onward.
    """
    
    def __init__(self, window_size: int = 3, timeout: float = 2.0):
        self.window_size = window_size
        self.timeout = timeout
        
        # Sender state
        self.send_base = 0      # Oldest unacknowledged frame
        self.next_seq = 0       # Next frame to send
        self.max_seq = 8        # Sequence numbers 0-7 (3 bits)
        
        # Buffers
        self.sent_frames: Dict[int, Tuple[Frame, float]] = {}
        self.received_acks: Set[int] = set()
        
        print(f"[WINDOW] Initialized with window size {window_size}")
    
    def can_send(self) -> bool:
        """Check if window has space for more frames."""
        # Calculate how many frames are "in flight"
        in_flight = (self.next_seq - self.send_base) % self.max_seq
        return in_flight < self.window_size
    
    def send_frame(self, frame: Frame) -> Optional[Frame]:
        """
        Attempt to send a frame.
        
        Returns: Frame with sequence number if sent, None if window closed
        """
        if not self.can_send():
            print(f"[WINDOW] Window full! {self.send_base} ≤ seq < {self.send_base + self.window_size}")
            return None
        
        # Assign sequence number
        frame.seq_num = self.next_seq
        self.next_seq = (self.next_seq + 1) % self.max_seq
        
        # Store with timestamp for timeout tracking
        self.sent_frames[frame.seq_num] = (frame, time.time())
        
        print(f"[WINDOW] Sent frame {frame.seq_num}, window: "
              f"[{self.send_base}..{(self.send_base + self.window_size - 1) % self.max_seq}]")
        
        return frame
    
    def receive_ack(self, ack_num: int):
        """
        Process incoming acknowledgment.
        
        ACK N means "I received frame N successfully"
        In Go-Back-N, this means all frames up to N are OK.
        """
        print(f"[WINDOW] Received ACK for frame {ack_num}")
        
        # Slide window forward
        while self.send_base != (ack_num + 1) % self.max_seq:
            if self.send_base in self.sent_frames:
                del self.sent_frames[self.send_base]
            self.send_base = (self.send_base + 1) % self.max_seq
            print(f"[WINDOW]   Slid window, new base: {self.send_base}")
    
    def check_timeouts(self) -> List[Frame]:
        """
        Check for frames that timed out (no ACK received).
        Returns list of frames to retransmit.
        """
        current_time = time.time()
        to_resend = []
        
        for seq_num, (frame, send_time) in list(self.sent_frames.items()):
            if current_time - send_time > self.timeout:
                print(f"[WINDOW] Frame {seq_num} timed out! Resending...")
                self.sent_frames[seq_num] = (frame, current_time)  # Update timestamp
                to_resend.append(frame)
        
        return to_resend



In [16]:

# Test Sliding Window
print("\n" + "=" * 60)
print("STEP 6: FLOW CONTROL (Sliding Window)")
print("=" * 60)

window = SlidingWindowProtocol(window_size=2)

# Try to send 4 frames with window size 2
frames_to_send = [
    Frame("aa:bb:cc:dd:ee:ff", "11:22:33:44:55:66", f"Message-{i}")
    for i in range(1, 5)
]

print("\n--- Sending frames ---")
for i, frame in enumerate(frames_to_send):
    result = window.send_frame(frame)
    if result is None:
        print(f"Frame {i+1}: BLOCKED (window full)")
        
        # Simulate receiving ACK to open window
        print("\n--- Simulating ACK to open window ---")
        window.receive_ack(window.send_base)  # ACK oldest frame
        
        # Try again
        result = window.send_frame(frame)
        if result:
            print(f"Frame {i+1}: SENT (after window opened)")
    else:
        print(f"Frame {i+1}: SENT")


STEP 6: FLOW CONTROL (Sliding Window)
[WINDOW] Initialized with window size 2

--- Sending frames ---
[WINDOW] Sent frame 0, window: [0..1]
Frame 1: SENT
[WINDOW] Sent frame 1, window: [0..1]
Frame 2: SENT
[WINDOW] Window full! 0 ≤ seq < 2
Frame 3: BLOCKED (window full)

--- Simulating ACK to open window ---
[WINDOW] Received ACK for frame 0
[WINDOW]   Slid window, new base: 1
[WINDOW] Sent frame 2, window: [1..2]
Frame 3: SENT (after window opened)
[WINDOW] Window full! 1 ≤ seq < 3
Frame 4: BLOCKED (window full)

--- Simulating ACK to open window ---
[WINDOW] Received ACK for frame 1
[WINDOW]   Slid window, new base: 2
[WINDOW] Sent frame 3, window: [2..3]
Frame 4: SENT (after window opened)
